# Lesson 15: Serving

Goes with the Lesson 15 page of the Forkcast course. An artifact that holds everything inference needs, an inference path that matches training, `torch.export`, FastAPI endpoints tested in-process, latency measurement, and the real server.

Runtime: CPU is enough.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .
!pip install -q fastapi httpx uvicorn

## 2. `forkcast/serve.py`

The pydantic request models are provided. Write `EtaPredictor.features/predict/save/load`, `export_program`, `load_exported`, `latency_stats` and the three endpoint bodies as the page builds them.

In [ ]:
%%writefile forkcast/serve.py
"""Lesson 15. Serving: a trained model becomes a service.

``Order``, ``latency_stats`` helpers' signatures and ``create_app``'s routes are fixed by the tests; the
preprocessing, the artifact, the export and the endpoint bodies are the exercises. ``deploy/app.py`` runs the
service with uvicorn, ``deploy/Dockerfile`` packages it.
"""

import statistics
import time
from pathlib import Path

import pandas as pd
import torch
from torch import Tensor, nn

from forkcast.model import TabularNet
from forkcast.preprocess import Normalizer
from forkcast.trips import CATEGORICAL, CONTINUOUS, ZONE_VOCAB

try:
    from pydantic import BaseModel, Field
except ImportError:                                   # the web extras are optional for the rest of the course
    BaseModel = object

    def Field(*args, **kwargs):
        return None


class Order(BaseModel):
    """One delivery as the app sends it. pydantic rejects anything outside these ranges before the model sees it."""
    trip_distance: float = Field(gt=0, le=50)
    pickup_hour: int = Field(ge=0, le=23)
    weekday: int = Field(ge=0, le=6)
    passenger_count: int = Field(ge=1, le=6)
    pu_zone: int = Field(ge=0, lt=ZONE_VOCAB)
    do_zone: int = Field(ge=0, lt=ZONE_VOCAB)


class EtaRequest(BaseModel):
    orders: list[Order] = Field(min_length=1, max_length=256)


class TextRequest(BaseModel):
    texts: list[str] = Field(min_length=1, max_length=256)


class ImageRequest(BaseModel):
    pixels: list        # (3, H, W) nested lists of floats in [0, 1]


class EtaPredictor:
    """Everything the ETA endpoint needs, in one object: the network and the normalizer statistics it was trained with.

    An artifact file holds ``{"state_dict", "mean", "std", "hidden"}``; loading rebuilds the model with no training code.
    """

    def __init__(self, model: TabularNet, normalizer: Normalizer, hidden=(64, 64)):
        self.model, self.normalizer, self.hidden = model.eval(), normalizer, tuple(hidden)

    def save(self, path: str | Path) -> None:
        """torch.save a dict with the model's state_dict, the normalizer's mean and std, and the hidden sizes."""
        raise NotImplementedError("Lesson 15, step 3: torch.save({...}, path)")

    @classmethod
    def load(cls, path: str | Path) -> "EtaPredictor":
        """Rebuild a TabularNet(4, zone_vocab=ZONE_VOCAB, hidden=...) and a Normalizer from the artifact, on the CPU."""
        raise NotImplementedError("Lesson 15, step 3: torch.load(path, map_location=\"cpu\", weights_only=True); rebuild model and normalizer; return cls(...)")

    def features(self, orders: list[dict]) -> tuple[Tensor, Tensor]:
        """Orders as dicts with the CONTINUOUS and CATEGORICAL keys -> (normalized continuous (N, 4), zone ids (N, 2) int64).

        Build a DataFrame from the dicts so the column order matches training exactly.
        """
        raise NotImplementedError("Lesson 15, step 2: frame = pd.DataFrame(orders); continuous via self.normalizer.transform; zones as long")

    def predict(self, orders: list[dict]) -> list[float]:
        """Minutes for a batch of orders, under torch.inference_mode(), as plain Python floats."""
        raise NotImplementedError("Lesson 15, step 2: with torch.inference_mode(): self.model(*self.features(orders)).tolist()")


def export_program(model: nn.Module, example_inputs: tuple, path: str | Path) -> None:
    """Freeze the model into a torch.export program with a dynamic batch dimension and save it as .pt2.

    ``batch = torch.export.Dim("batch")``; dynamic_shapes marks dimension 0 of every example input as ``batch``;
    ``torch.export.export(model, example_inputs, dynamic_shapes=...)`` then ``torch.export.save(program, path)``.
    """
    raise NotImplementedError("Lesson 15, step 4: Dim, dynamic_shapes, torch.export.export, torch.export.save")


def load_exported(path: str | Path) -> nn.Module:
    """Load a .pt2 program and return a callable module: ``torch.export.load(path).module()``."""
    raise NotImplementedError("Lesson 15, step 4: torch.export.load(path).module()")


def latency_stats(fn, payloads: list, repeats: int = 3) -> dict:
    """Call fn(payload) for every payload ``repeats`` times; return {"p50_ms", "p95_ms", "mean_ms", "n"}.

    Use time.perf_counter around each call; statistics.quantiles(times, n=20)[9] is the median and [18] the 95th percentile.
    """
    raise NotImplementedError("Lesson 15, step 6: time each call with perf_counter, then quantiles")


def create_app(eta: EtaPredictor, sentiment=None, dish=None):
    """The Forkcast ML service: GET /health, POST /eta, POST /sentiment, POST /dish.

    ``sentiment`` is (model, vocab) from Lesson 11 or None; ``dish`` is (model, class_names) from Lesson 7 or None.
    FastAPI validates request bodies against the pydantic models above and answers 422 when they do not match.
    """
    from fastapi import FastAPI, HTTPException

    app = FastAPI(title="Forkcast ML service")

    @app.get("/health")
    def health():
        return {"status": "ok", "torch": torch.__version__}

    @app.post("/eta")
    def predict_eta(request: EtaRequest):
        """Return {"minutes": [...]} with one float per order, in order."""
        raise NotImplementedError("Lesson 15, step 5: eta.predict([o.model_dump() for o in request.orders])")

    @app.post("/sentiment")
    def predict_sentiment(request: TextRequest):
        """Return {"results": [{"label": "positive"|"negative", "p_positive": float}, ...]} or 503 if no model."""
        raise NotImplementedError("Lesson 15, step 5: tokenize, encode, pad_batch, softmax; 503 when sentiment is None")

    @app.post("/dish")
    def predict_dish(request: ImageRequest):
        """Return {"label": str, "confidence": float} for one (3, H, W) image, 400 if the shape is wrong, 503 if no model."""
        raise NotImplementedError("Lesson 15, step 5: torch.tensor(pixels); check 3 dims and 3 channels; softmax; class name")

    return app

## 3. A trained ETA model, and the inference path

In [ ]:
import importlib, os, torch
from torch import nn
from fastapi.testclient import TestClient
from forkcast import serve, trips, train, text, vision
from forkcast.model import TabularNet
from forkcast.preprocess import Normalizer
importlib.reload(serve)

frame = trips.load_trips(verbose=False)
tr, va, _ = trips.split_frame(frame, seed=0)
norm = Normalizer().fit(torch.tensor(tr[trips.CONTINUOUS].values, dtype=torch.float32))
tl, vl = trips.make_loaders(tr, va, norm, batch_size=256)
torch.manual_seed(0)
model = TabularNet(4, hidden=(32, 32))
train.fit_loader(model, tl, vl, epochs=5, lr=3e-3)
eta = serve.EtaPredictor(model, norm, hidden=(32, 32))
orders = tr.head(8)[trips.CONTINUOUS + trips.CATEGORICAL].astype({"pickup_hour": int, "weekday": int, "passenger_count": int}).to_dict(orient="records")
print(orders[0])
print("batched:", [round(v, 2) for v in eta.predict(orders[:3])], "| single:", [round(eta.predict([o])[0], 2) for o in orders[:3]])

## 4. The artifact

In [ ]:
importlib.reload(serve)
os.makedirs("artifacts", exist_ok=True)
eta = serve.EtaPredictor(model, norm, hidden=(32, 32))
eta.save("artifacts/eta.pt")
print("artifact:", round(os.path.getsize("artifacts/eta.pt") / 1024, 1), "kB; keys", sorted(torch.load("artifacts/eta.pt", weights_only=True)))
loaded = serve.EtaPredictor.load("artifacts/eta.pt")
print("same answers after reload:", torch.allclose(torch.tensor(loaded.predict(orders)), torch.tensor(eta.predict(orders)), atol=1e-5))

## 5. Export

In [ ]:
importlib.reload(serve)
cont, cat = eta.features(orders)
serve.export_program(eta.model, (cont[:4], cat[:4]), "artifacts/eta.pt2")
exported = serve.load_exported("artifacts/eta.pt2")
with torch.inference_mode():
    for n in (1, 3, 8):
        print(f"batch {n}: max difference", (exported(cont[:n], cat[:n]) - eta.model(cont[:n], cat[:n])).abs().max().item())

## 6. The API, tested in-process

A sentiment model from Lesson 11 and a shape classifier from Lesson 7 stand in for the real dish model; the capstone swaps in Lesson 9's ResNet.

In [ ]:
importlib.reload(serve)
reviews = text.load_reviews()
vocab = text.Vocab([text.tokenize(t) for t in reviews["text"]])
r_tl, r_vl = text.make_text_loaders(reviews["text"][:640], reviews["label"][:640], reviews["text"][640:], reviews["label"][640:], vocab, batch_size=32)
torch.manual_seed(0); bag = text.BagOfEmbeddings(len(vocab), dim=32, hidden=32)
train.fit_v2(bag, r_tl, r_vl, epochs=6, lr=3e-3, loss_fn=nn.CrossEntropyLoss())
s_tl, s_vl = vision.shape_loaders(640, 128)
torch.manual_seed(0); cnn = vision.SmallCNN()
train.fit_v2(cnn, s_tl, s_vl, epochs=3, lr=1e-3, loss_fn=nn.CrossEntropyLoss())
torch.save({"state_dict": bag.state_dict(), "itos": vocab.itos, "dim": 32, "hidden": 32}, "artifacts/sentiment.pt")
torch.save({"state_dict": cnn.state_dict(), "classes": vision.SHAPE_CLASSES, "size": 32}, "artifacts/dish.pt")

client = TestClient(serve.create_app(eta, sentiment=(bag, vocab), dish=(cnn, vision.SHAPE_CLASSES)))
print(client.get("/health").json())
print(client.post("/eta", json={"orders": orders[:2]}).json())
bad = dict(orders[0]); del bad["trip_distance"]
r = client.post("/eta", json={"orders": [bad]}); print(r.status_code, r.json()["detail"][0])
print(client.post("/sentiment", json={"texts": ["Delicious, will order again.", "Cold, soggy and an hour late."]}).json())
images, labels = vision.make_shapes(2, seed=3)
print(client.post("/dish", json={"pixels": images[0].tolist()}).json(), "| true:", vision.SHAPE_CLASSES[labels[0]])
r = client.post("/dish", json={"pixels": images[0].permute(1, 2, 0).tolist()}); print(r.status_code, r.json())

## 7. Latency

In [ ]:
importlib.reload(serve)
single = serve.latency_stats(lambda o: eta.predict([o]), orders, repeats=5)
batched = serve.latency_stats(lambda os: eta.predict(os), [orders * 8], repeats=5)
http = serve.latency_stats(lambda o: client.post("/eta", json={"orders": [o]}), orders, repeats=3)
print(f"one order per call: p50 {single['p50_ms']:.2f} ms, p95 {single['p95_ms']:.2f} ms")
print(f"64 orders per call: {batched['mean_ms']:.2f} ms total, {batched['mean_ms'] / 64:.3f} ms per order")
print(f"HTTP round trip, one order: p50 {http['p50_ms']:.2f} ms")

### Break it on purpose: a fresh normalizer

In [ ]:
fresh = Normalizer().fit(torch.tensor(__import__("pandas").DataFrame(orders)[trips.CONTINUOUS].values, dtype=torch.float32))
wrong = serve.EtaPredictor(model, fresh, hidden=(32, 32))
print("trained statistics:", eta.normalizer.mean.tolist()); print("fresh statistics:  ", fresh.mean.tolist())
print("right:", [round(v, 1) for v in eta.predict(orders[:4])]); print("wrong:", [round(v, 1) for v in wrong.predict(orders[:4])], "<- status 200, no error, minutes off")

## 8. The real server

`deploy/app.py` loads the artifacts you just wrote. Visit the printed URL + `/docs` if your environment exposes the port.

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "deploy.app:app", "--host", "127.0.0.1", "--port", "8000"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
print(requests.get("http://127.0.0.1:8000/health").json())
print(requests.post("http://127.0.0.1:8000/eta", json={"orders": orders[:2]}).json())
print(requests.post("http://127.0.0.1:8000/sentiment", json={"texts": ["Never again."]}).json())
server.terminate()

## 9. Definition of done

Expected: `7 passed`, then `111 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_15.py -q

In [ ]:
!pytest -m fast -q

## 10. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 15: serving"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main